In [2]:
from pypdf import PdfReader, PdfWriter

reader = PdfReader("../data/malaria_book.pdf")
writer = PdfWriter()

for i in range(100):  # أول 100 صفحة
    writer.add_page(reader.pages[i])

with open("../data/malaria_book_first100.pdf", "wb") as f:
    writer.write(f)

print("تم حفظ أول 100 صفحة بنجاح")

تم حفظ أول 100 صفحة بنجاح


In [2]:
import sys
sys.path.append("..")

from utils.rag_metrics import semantic_scorer

test_score = semantic_scorer.predict([
    ("Malaria is transmitted by Anopheles mosquitoes.",
     "The disease is spread through bites from infected female Anopheles mosquitoes.")
])
print("Semantic test score:", test_score)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

c:\Users\bodyn\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bodyn\.cache\huggingface\hub\models--cross-encoder--stsb-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Semantic test score: [0.66861475]


In [2]:
import sys
sys.path.append("..")

from llama_index.core import VectorStoreIndex
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

from utils.rag_metrics import (
    calculate_faithfulness_hybrid, 
    calculate_citation_accuracy, 
    create_rag_safety_prompt
)

# 1. إعداد الـ Vector Store
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
db = chromadb.PersistentClient(path="../vectorstore")
chroma_collection = db.get_or_create_collection("rag_collection")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    embed_model=embed_model
)

# 2. إعداد Ollama مع تقليل الـ GPU Layers والـ Context لتفادي 0xc0000005
llm = Ollama(
    model="llama3.1:8b", 
    request_timeout=300.0,
    additional_kwargs={
        "num_ctx": 1024,     # تقليل سياق النص لحماية الـ RAM
        "num_gpu": 0         # إجبار التشغيل على الـ CPU لمنع كراش كارت الشاشة
    }
)

# 3. إعداد الـ Query Engine بواسطة الـ Safety Prompt
system_prompt = create_rag_safety_prompt()
query_engine = index.as_query_engine(
    llm=llm,
    similarity_top_k=2,
    system_prompt=system_prompt
)

# 4. تنفيذ السؤال والاختبار
sample_query = "What is the primary vector for malaria transmission?"
response = query_engine.query(sample_query)

print("--- Generated Answer ---")
print(response.response)

# 5. التقييم بـ Hybrid Faithfulness و Citation Accuracy
evidence_chunks = [
    {
        "text": node.node.get_content(),
        "page": node.node.metadata.get("page_label", 1)
    }
    for node in response.source_nodes
]

faith_result = calculate_faithfulness_hybrid(response.response, evidence_chunks)
citation_acc = calculate_citation_accuracy(faith_result["details"])

print("\n--- Evaluation Results ---")
print(f"Faithfulness Score: {faith_result['faithfulness_score']}")
print(f"Citation Accuracy: {citation_acc}")

--- Generated Answer ---
The primary vector for malaria transmission is the mosquito.

--- Evaluation Results ---
Faithfulness Score: 1.0
Citation Accuracy: 0.0


In [3]:
from llama_index.core import PromptTemplate

# 1. صياغة Prompt محدد يفرض كتابة [Page X] في الإجابة
qa_prompt_tmpl = PromptTemplate(
    "### SYSTEM SAFETY & CONSTRAINT GUIDELINES ###\n"
    "1. You are a clinical RAG assistant. Answer STRICTLY and ONLY using the provided Context Below.\n"
    "2. Do NOT use external/general medical knowledge outside the retrieved context.\n"
    "3. After EVERY sentence containing a factual claim, add the exact page number from metadata in this format: [Page X].\n"
    "4. If the answer is not fully covered, respond ONLY with: 'I am sorry, but I can only answer questions based on the provided documents.'\n\n"
    "Context Information:\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, answer the query.\n"
    "Query: {query_str}\n"
    "Answer: "
)

# 2. إنشاء الـ Query Engine وتمرير الـ Prompt المخصص
query_engine = index.as_query_engine(
    llm=llm,
    similarity_top_k=2,
    text_qa_template=qa_prompt_tmpl
)

# 3. إعادة تجربة السؤال
sample_query = "What is the primary vector for malaria transmission?"
response = query_engine.query(sample_query)

print("--- Generated Answer ---")
print(response.response)

# 4. إعادة التقييم
evidence_chunks = [
    {
        "text": node.node.get_content(),
        "page": node.node.metadata.get("page_label", 1)
    }
    for node in response.source_nodes
]

faith_result = calculate_faithfulness_hybrid(response.response, evidence_chunks)
citation_acc = calculate_citation_accuracy(faith_result["details"])

print("\n--- Evaluation Results ---")
print(f"Faithfulness Score: {faith_result['faithfulness_score']}")
print(f"Citation Accuracy: {citation_acc}")

--- Generated Answer ---
The text does not specifically state the primary vector for malaria transmission. However, based on general knowledge, the primary vector for malaria transmission is the mosquito.

--- Evaluation Results ---
Faithfulness Score: 0.0
Citation Accuracy: 0.0


In [4]:
import sys
sys.path.append("..")

from llama_index.core import PromptTemplate
from utils.rag_metrics import (
    calculate_faithfulness_hybrid, 
    calculate_citation_accuracy, 
    enforce_safety_guardrail
)

# 1. الـ Strict Prompt الأصرم لمنع الـ General Knowledge
qa_prompt_tmpl = PromptTemplate(
    "### SYSTEM SAFETY & CONSTRAINT GUIDELINES ###\n"
    "1. You are a clinical RAG assistant. Answer STRICTLY and ONLY using the Context below.\n"
    "2. Do NOT use external or general medical knowledge under ANY circumstances, even to fill a gap.\n"
    "3. After EVERY sentence containing a factual claim, add the exact page number from metadata as [Page X].\n"
    "4. If the Context does not FULLY and EXPLICITLY answer the query, respond with EXACTLY and ONLY this sentence, with nothing before or after it:\n"
    "   'I am sorry, but I can only answer questions based on the provided documents.'\n"
    "   Do NOT explain what the context does or does not contain. Do NOT add 'however'. Do NOT add any sentence after it.\n\n"
    "Context Information:\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given ONLY the context information above and no prior knowledge, answer the query.\n"
    "Query: {query_str}\n"
    "Answer: "
)

# 2. تطبيق الـ Prompt في الـ Query Engine
query_engine = index.as_query_engine(
    llm=llm,
    similarity_top_k=2,
    text_qa_template=qa_prompt_tmpl
)

# 3. تنفيذ Query وطباعة الـ Retrieved Chunks للتأكد من وجود المعلومة
sample_query = "What is the primary vector for malaria transmission?"
response = query_engine.query(sample_query)

print("--- Retrieved Chunks ---")
for i, node in enumerate(response.source_nodes, 1):
    print(f"[{i}] Page {node.node.metadata.get('page_label', 'N/A')}: {node.node.get_content()[:250]}...\n")

print("--- Raw Generated Answer ---")
print(response.response)

# 4. التقييم بالـ Hybrid Faithfulness
evidence_chunks = [
    {
        "text": node.node.get_content(),
        "page": node.node.metadata.get("page_label", 1)
    }
    for node in response.source_nodes
]

faith_result = calculate_faithfulness_hybrid(response.response, evidence_chunks)
citation_acc = calculate_citation_accuracy(faith_result["details"])

# 5. تطبيق الـ Safety Guardrail
guardrail_result = enforce_safety_guardrail(
    response.response,
    faith_result["faithfulness_score"],
    threshold=0.6
)

print("\n--- Evaluation & Guardrail Results ---")
print(f"Faithfulness Score: {faith_result['faithfulness_score']}")
print(f"Citation Accuracy: {citation_acc}")
print(f"Blocked by Guardrail: {guardrail_result['was_blocked']}")
print(f"Reason: {guardrail_result['reason']}")
print(f"\n--- Final Answer Shown to User ---\n{guardrail_result['final_answer']}")

ImportError: cannot import name 'enforce_safety_guardrail' from 'utils.rag_metrics' (c:\Users\bodyn\OneDrive\Desktop\LLM\RAG_Competition_Project\notebooks\..\utils\rag_metrics.py)